In [11]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

In [12]:
MODEL_C = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer_c = AutoTokenizer.from_pretrained(MODEL_C)
if tokenizer_c.pad_token is None:
    tokenizer_c.pad_token = tokenizer_c.eos_token
use_qlora = torch.cuda.is_available()
if use_qlora:
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    base_c = AutoModelForCausalLM.from_pretrained(
        MODEL_C,
        quantization_config=quant_config,
        device_map="auto",
    )
    base_c = prepare_model_for_kbit_training(base_c)
else:
    base_c = AutoModelForCausalLM.from_pretrained(MODEL_C)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model_c = get_peft_model(base_c, lora_config)
model_c.print_trainable_parameters()

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 21995.69it/s]


trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


In [13]:
# TODO(student 3): build 40–80 high-quality technical-support conversations
# in messages format. Include successful troubleshooting, uncertainty,
# tool-result synthesis, and human-escalation examples.
sft_sample_rows = [
    {
        "messages": [
            {
                "role": "system",
                "content": "You are a technical-support specialist. Give grounded, concise, and safe answers.",
            },
            {
                "role": "user",
                "content": "The API started returning HTTP 503 after deployment. What should I check?",
            },
            {
                "role": "assistant",
                "content": "Check the service health endpoint, deployment logs, environment variables, and database connectivity. If the service is unhealthy, roll back the deployment and preserve the logs for investigation.",
            },
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": "Answer only from verified technical context. State clearly when information is missing.",
            },
            {
                "role": "user",
                "content": "Which port must be exposed for the payment service?",
            },
            {
                "role": "assistant",
                "content": "I cannot verify the port because no payment-service configuration or deployment documentation was provided. Check the service configuration before exposing a port rather than guessing.",
            },
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": "Synthesize tool results and escalate high-risk production incidents to a human.",
            },
            {
                "role": "user",
                "content": "The diagnostic tool reports a failed migration and possible production database corruption.",
            },
            {
                "role": "assistant",
                "content": "The tool result indicates a high-risk database incident. Stop automated recovery and avoid further writes. Preserve the migration logs and escalate immediately to the database administrator or on-call engineer.",
            },
        ]
    },
]


In [14]:
all_sft_data = Dataset.from_list(sft_sample_rows)

data_split = all_sft_data.train_test_split(
    test_size=0.2,
    seed=42,
)

sft_dataset = {
    "train": data_split["train"],
    "validation": data_split["test"],
}

In [15]:
def format_for_sft(example):
    return {
        "text": tokenizer_c.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

sft_train = sft_dataset["train"].map(format_for_sft)
sft_val = sft_dataset["validation"].map(format_for_sft)

Map: 100%|██████████| 1/1 [00:00<00:00, 656.49 examples/s]


In [16]:
sft_args = SFTConfig(
    output_dir="models/support_adapter",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataset_text_field="text",
    max_length=512,
    packing=False,
    report_to="none",
    save_total_limit=1,
)

trainer_c = SFTTrainer(
    model=model_c,
    args=sft_args,
    train_dataset=sft_train,
    eval_dataset=sft_val,
    processing_class=tokenizer_c,
)
trainer_c.train()
model_c.save_pretrained("models/support_adapter")
tokenizer_c.save_pretrained("models/support_adapter")

Truncating train dataset: 100%|██████████| 2/2 [00:00<00:00, 1201.29 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 2/2 [00:00<00:00, 1236.71 examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 1/1 [00:00<00:00, 930.21 examples/s]
/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,No log,4.260853,3.010124,152.000000,0.333333
2,No log,4.266138,3.019140,304.000000,0.333333


/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[transformers] mtime may not be reliable on this filesystem, falling back to numerical ordering


('models/support_adapter/tokenizer_config.json',
 'models/support_adapter/chat_template.jinja',
 'models/support_adapter/tokenizer.json')